In [ ]:
import torch
import random
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class TokenStreamDataset(torch.utils.data.Dataset):
    def __init__(self, source, seq_len=16):
        super().__init__()

        self.source = source
        self.seq_len = seq_len
        self.raw = open(source, "r").read()

        self.vocab = sorted(list(set(self.raw)))
        self.encode = lambda string: [self.vocab.index(s) for s in string]
        self.decode = lambda tokens: "".join([self.vocab[t] for t in tokens])

        self.tokens = self.encode(self.raw)
    
    def __getitem__(self, index):
        x = self.tokens[index:index+self.seq_len]
        y = self.tokens[index+1:index+self.seq_len+1]
        return torch.tensor(x), torch.tensor(y)

    def __len__(self):
        return len(self.tokens) - self.seq_len
    
dataset_train = TokenStreamDataset(
    source="wikipedia.txt",
    seq_len=128
)

dataloader_train = torch.utils.data.DataLoader(
    dataset=dataset_train,
    batch_size=2048,
    shuffle=True,
    num_workers=8,
    pin_memory=True
)

dataset_train[0], len(dataloader_train)

In [ ]:
class RNNLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_size = hidden_size

        self.embedding = nn.Parameter(torch.randn(vocab_size, embedding_dim))

        self.W_x = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.1)
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)

        self.linear = nn.Linear(in_features=self.hidden_size, out_features=self.vocab_size)
    
    def forward(self, X):
        X = self.embedding[X]
        h = torch.zeros(X.shape[0], self.hidden_size, device=X.device)
        hs = []

        for t in range(0, X.shape[1]):
            x_t = X[:, t, :]
            h = torch.tanh(
                x_t @ self.W_x.T +
                h @ self.W_h.T
            )
            hs.append(h)
        
        hs = torch.stack(hs, dim=0).transpose(0, 1)
        logits = self.linear(hs)

        return logits
    
dummy = RNNLM(
    vocab_size=len(dataset_train.vocab),
    embedding_dim=32,
    hidden_size=32
)
dummy(dataset_train[0][0].unsqueeze(dim=0)).shape

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RNNLM(
    vocab_size=len(dataset_train.vocab),
    embedding_dim=64,
    hidden_size=256
).to(device)

optimizer = torch.optim.Adam(
    params=model.parameters(),
    lr=5e-4
)

criterion = nn.CrossEntropyLoss()

num_epochs = 64

In [ ]:
for epoch in range(1, num_epochs+1):
    for batch, (X, y) in enumerate(dataloader_train, 1):
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        y_logits = model(X)
        loss = criterion(y_logits.flatten(start_dim=0, end_dim=1), y.flatten())
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()

        if batch == 1 or batch % 10 == 0 or batch == len(dataloader_train):
            print(f"Epoch [{epoch}/{num_epochs}] Batch [{batch}/{len(dataloader_train)}] Loss: {loss:4f}", end="\r")
        if batch == len(dataloader_train):
            print(end="\n")

In [ ]:
def predict(model, prompt, dataset, temperature=0.75, top_k=10):
    model.eval()
    prompt = torch.tensor(dataset.encode(prompt)).unsqueeze(dim=0)
    
    with torch.inference_mode():
        y_logits = model(prompt)[0, -1].detach().cpu()

    probs = F.softmax(y_logits / temperature, dim=0).sort(descending=True)
    probs = {
        dataset.decode([index]): round(value.item(), 4)
        for (value, index) in zip(probs.values, probs.indices)
    }
    probs = dict(list(probs.items())[:top_k])

    return probs

def generate(model, prompt, dataset, **kwargs):
    probs = predict(model, prompt, dataset, **kwargs)
    choice = random.choices(list(probs.keys()), list(probs.values()))
    return choice[0]

In [ ]:
text = "The"
for i in range(500):
    prediction = generate(
        model=model,
        prompt=text,
        dataset=dataset_train,
        temperature=0.75,
        top_k=10
    )
    text += prediction

for index, word in enumerate(text.split(), 1):
    print(word, end=" ")
    if index % 10 == 0:
        print(end="\n")